In [ ]:
import pandas as pd
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# ─── Paramètres de MinIO ───────────────────────────────────────────
MINIO_ENDPOINT   = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
BUCKET_NAME      = "offreemploi"
S3_OUTPUT_PREFIX = "tableaux_nettoyes"

# ─── Connexion MinIO ───────────────────────────────────────────────
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
    verify=False
)

# ─── Fonction de nettoyage et d'export vers MinIO ─────────────────
def nettoyer_et_uploader_csv(csv_path, nom_fichier_xlsx):
    try:
        df = pd.read_csv(csv_path, low_memory=False)

        # Nettoyage
        seuil = len(df) * 0.8
        df = df.dropna(axis=1, thresh=seuil)                    # Supprimer colonnes vides
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]    # Supprimer colonnes sans nom
        df = df.dropna(how='all')                               # Supprimer lignes entièrement vides
        df = df.drop_duplicates()                               # Supprimer doublons

        # Normaliser les noms de colonnes
        df.columns = (
            df.columns.str.strip()
                      .str.replace(" ", "_")
                      .str.replace("’", "_")
                      .str.replace("'", "_")
                      .str.replace("__", "_")
                      .str.upper()
        )

        # Exporter en Excel vers mémoire
        buffer = BytesIO()
        df.to_excel(buffer, index=False)
        buffer.seek(0)

        # Upload sur MinIO
        s3_key = f"{S3_OUTPUT_PREFIX}/{nom_fichier_xlsx}"
        s3_client.upload_fileobj(buffer, BUCKET_NAME, s3_key)
        print(f"✅ Fichier nettoyé chargé sur MinIO : {BUCKET_NAME}/{s3_key}")

    except ClientError as e:
        print(f"❌ Erreur MinIO : {e}")
    except Exception as e:
        print(f"❌ Erreur générale : {e}")

# ─── Exemple d’utilisation ─────────────────────────────────────────
if __name__ == "__main__":
    # ⚠️ Chemin vers votre fichier CSV local
    csv_local = r"C:\Users\a_doumbia\Desktop\DOUMBIA_PC\COLLECTE_JOURNALIERE_offre_emploi\donnees_offres_nettoyees.csv"
    xlsx_nom = "offres_emploi_nettoyees.xlsx"
    nettoyer_et_uploader_csv(csv_local, xlsx_nom)
